# Классификация: конспект

Этот конспект связывает основные идеи логистической регрессии, SVM и метрик. Он предназначен для повторения материала перед практикумом: определения сопровождаются объяснениями и короткими примерами.

[Подробное изложение с выводами](2.%20Classification%20%E2%80%94%20theory%20detailed.ipynb) · [Практикум](2.%20Classification%2C%20Logistic%20Regression%2C%20Validation.ipynb)

## 1. Постановка задачи и три вида предсказаний
Даны объекты с признаками $x_i\in\mathbb R^d$ и метками $y_i\in\{0,1\}$. Нужно научиться предсказывать класс **новых** объектов. Класс 1 называют положительным; в задаче поиска спама это может быть именно спам.

Не путайте три результата модели:

| Результат | Обозначение | Что означает |
|---|---|---|
| Score | $f(x)\in\mathbb R$ | Численная оценка: больше — сильнее предпочтение класса 1 |
| Вероятность | $p(x)\in[0,1]$ | Оценка условной вероятности $P(y=1\mid x)$ |
| Метка | $\hat y_t=\mathbf1\{p(x)\ge t\}$ | Решение при выбранном пороге $t$ |

Score не обязательно является вероятностью. При $p=0.4$ порог 0.5 даёт класс 0, а порог 0.3 — класс 1: модель та же, решение другое.

Линейная модель вычисляет $z=w^\top x+b$. При решении по знаку $z$ граница $w^\top x+b=0$ — прямая в двух измерениях и гиперплоскость в общем случае. Вектор $w$ задаёт направление нормали, $b$ — сдвиг границы.

## 2. Логистическая регрессия
### 2.1. Как линейная оценка становится вероятностью
Используем сигмоиду:
$$p(x)=\sigma(w^\top x+b),\qquad \sigma(z)=\frac1{1+e^{-z}}.$$
При $z=0$ вероятность равна 0.5; при $z=2$ — примерно 0.88. Сигмоида монотонна, поэтому при пороге 0.5 граница по-прежнему линейна: $w^\top x+b=0$.

Эквивалентное предположение модели:
$$\log\frac{p}{1-p}=w^\top x+b.$$
Именно **логарифм шансов**, а не сама вероятность, линейно зависит от признаков.

### 2.2. Откуда берётся log loss
Предполагаем $y_i\mid x_i\sim\operatorname{Bernoulli}(p_i)$. Вероятность наблюдённой метки равна $p_i$ для класса 1 и $1-p_i$ для класса 0. Для независимых наблюдений эти вероятности перемножаются.

Максимизация правдоподобия эквивалентна минимизации среднего отрицательного логарифма:
$$J=\frac1n\sum_i\ell_i,\qquad
\ell_i=-y_i\log p_i-(1-y_i)\log(1-p_i).$$
Это бинарная кросс-энтропия, или log loss. Для положительного объекта прогноз $p=0.9$ даёт потерю 0.105, а $p=0.1$ — 2.303. Уверенная ошибка штрафуется сильнее.

В отличие от числа неправильных меток, эта потеря непрерывно меняется при изменении score. Поэтому её удобно оптимизировать градиентными методами.

### 2.3. Градиент и L2-регуляризация
При $z_i=w^\top x_i+b$ ту же потерю можно записать как
$$\ell_i=\log(1+e^{z_i})-y_i z_i,\qquad
\frac{\partial\ell_i}{\partial z_i}=\sigma(z_i)-y_i=p_i-y_i.$$
Для устойчивого вычисления удобно взять $s_i=2y_i-1$ и использовать `np.logaddexp(0, -s_i*z_i)`: буквальное вычисление экспоненты может переполниться.

Добавляя L2-штраф без штрафования сдвига, получаем
$$J_\lambda=\frac1n\sum_i\ell_i+\frac\lambda2\|w\|^2,$$
$$\nabla_wJ_\lambda=\frac1nX^\top(p-y)+\lambda w,\qquad
\frac{\partial J_\lambda}{\partial b}=\frac1n\sum_i(p_i-y_i).$$
Здесь строки $X$ — объекты. Шаг градиентного спуска вычитает из параметров градиент, умноженный на шаг $\eta$.

Большая $\lambda$ сильнее ограничивает коэффициенты; $\eta$ регулирует шаг оптимизации. Масштабы признаков влияют на смысл L2, поэтому признаки обычно стандартизуют по статистикам **train**.

### 2.4. Почему веса растут после accuracy = 1
Если данные строго линейно разделимы, умножение разделяющих весов и сдвига на $c>1$ сохраняет правильные классы, но увеличивает уверенность. При $c\to\infty$ log loss стремится к нулю, не достигая его при конечных параметрах.

Поэтому без регуляризации конечного минимума на строго разделимых данных нет. Идеальная accuracy не означает, что минимизирована log loss. L2 вводит цену за большие веса. Это основное наблюдение второго упражнения.

## 3. SVM: разделение с широким зазором
### 3.1. Отступ и геометрия
Для SVM используем метки $s_i=2y_i-1\in\{-1,+1\}$. Функциональный отступ
$$m_i=s_i(w^\top x_i+b)$$
положителен при правильной классификации. Чтобы получить подписанное расстояние до границы, делим его на $\|w\|$.

На разделимых данных **hard-margin SVM** выбирает самый широкий зазор:
$$\min_{w,b}\frac12\|w\|^2\quad\text{при}\quad m_i\ge1.$$
Нормировка $m_i\ge1$ фиксирует масштаб параметров. Расстояние между опорными гиперплоскостями $w^\top x+b=\pm1$ равно $2/\|w\|$, поэтому минимизация нормы увеличивает ширину зазора.

### 3.2. Когда классы пересекаются
**Soft-margin SVM** допускает нарушения и штрафует их:
$$\min_{w,b}\frac12\|w\|^2+C\sum_i\max(0,1-m_i).$$
Функция $\max(0,1-m)$ называется hinge loss. Она равна нулю при $m\ge1$, положительна внутри зазора и больше 1 при отрицательном отступе. Например, $m=0.4$ означает правильную классификацию, но потерю 0.6.

Большое $C$ делает нарушения дороже. Это противоположное направление относительно $\lambda$ в нашей записи L2. Для критерия «норма плюс $C$ × сумма потерь» переход к средней потере даёт $\lambda=1/(Cn)$.

**Сравнение:** логистическая потеря $\log(1+e^{-m})$ продолжает уменьшаться при росте любого конечного отступа; hinge loss обнуляется после $m=1$. Логистическая регрессия моделирует вероятность, а обычный SVM выдаёт score, не вероятность.

## 4. Нелинейные границы: признаки и ядра
Линейность модели относится к признакам, которые поданы ей на вход. Если заменить $x$ на $\phi(x)$, граница $w^\top\phi(x)+b=0$ может быть нелинейной в исходных координатах.

**XOR:** класс 1 соответствует разным знакам $x_1,x_2$. Произведение $u=x_1x_2$ отрицательно ровно в этих квадрантах. Значит, правило становится линейным в новом признаке: $y=\mathbf1\{u<0\}$ (значения на осях несущественны при непрерывной генерации).

Полином степени 2 содержит $(x_1,x_2,x_1^2,x_1x_2,x_2^2)$. Существенно именно **взаимодействие $x_1x_2$**: одних квадратов вместе с исходными признаками недостаточно для точного XOR.

SVM может работать в новом пространстве через ядро $K(x,x')=\langle\phi(x),\phi(x')\rangle$, не строя $\phi$ явно. Для **RBF-ядра**
$$K(x,x')=\exp(-\gamma\|x-x'\|^2).$$
Большое $\gamma$ делает влияние объектов более локальным и позволяет более сложную границу. $C$ регулирует штраф за нарушения. Оба параметра выбирают по validation; расстояния требуют внимания к масштабу признаков.

## 5. Метрики для фиксированного порога
Строки матрицы ошибок — истинные классы, столбцы — предсказанные:

| | Предсказан 0 | Предсказан 1 |
|---|---:|---:|
| Истинный 0 | TN | FP: ложная тревога |
| Истинный 1 | FN: пропуск | TP |

| Метрика | Формула | Вопрос |
|---|---|---|
| Accuracy | $(TP+TN)/n$ | Какая доля всех ответов правильна? |
| Precision | $TP/(TP+FP)$ | Какая доля срабатываний правильна? |
| Recall | $TP/(TP+FN)$ | Какая доля положительных объектов найдена? |
| F1 | $2TP/(2TP+FP+FN)$ | Насколько одновременно велики precision и recall? |

Пример: $TP=15$, $FN=5$, $FP=10$, $TN=70$. Получаем accuracy 0.85, precision 0.60, recall 0.75, F1 около 0.667. Знаменатели различаются: precision рассматривает **предсказанные** положительные, recall — **истинные**.

При дисбалансе accuracy может вводить в заблуждение: постоянный ответ 0 даёт 99% правильных меток, если положительных только 1%, но не находит ни одного события. F1 тоже не универсальна: она не учитывает TN и не заменяет явную стоимость ошибок. При нулевых знаменателях нужно задать соглашение вычисления метрик.

### Выбор порога
Снижая порог, увеличиваем число положительных решений: recall не уменьшается, но precision не обязана меняться монотонно.

При корректной вероятности $p$ и ценах ошибок $c_{FP},c_{FN}$ выбираем класс 1, если его ожидаемая цена $c_{FP}(1-p)$ не больше $c_{FN}p$. Отсюда
$$t^*=\frac{c_{FP}}{c_{FP}+c_{FN}}.$$
Порог 0.5 соответствует одинаковой цене ошибок. Подбирать порог по наблюдаемому качеству нужно на validation, а не на test.

## 6. Метрики ранжирования и вероятностей
### ROC AUC и AP
Этим метрикам передают **scores или вероятности**, а не бинарные метки: важно, в каком порядке модель ставит объекты.

- **ROC-кривая:** $TPR=TP/(TP+FN)$ по вертикали против $FPR=FP/(FP+TN)$ по горизонтали при изменении порога. ROC AUC — вероятность поставить случайный положительный объект выше случайного отрицательного, плюс половина вероятности совпадения scores.
- **PR-кривая:** precision против recall. **Average precision (AP)** суммирует precision с весами приростов recall: $AP=\sum_k(R_k-R_{k-1})P_k$. Это не обязательно площадь, полученная методом трапеций. Базовый ориентир precision зависит от доли положительных объектов.

Например, порядок меток по убыванию score $1,0,1,0$ даёт ROC AUC $3/4$: из четырёх положительно-отрицательных пар три упорядочены верно. AP равна $\tfrac12\cdot1+\tfrac12\cdot\tfrac23=5/6$.

Строго возрастающее преобразование scores сохраняет ранжирование и эти метрики. Но оно может изменить смысл чисел как вероятностей.

### Log loss и калибровка
Log loss оценивает вероятности и сильно штрафует уверенные ошибки. Для меток $(1,0)$ прогнозы $(0.6,0.4)$ и $(0.99,0.98)$ имеют одинаковую идеальную ROC AUC, но средняя log loss примерно 0.511 и 1.961 соответственно.

Калиброванная вероятность 0.7 означает, что среди объектов с таким прогнозом событие происходит примерно в 70% случаев. Высокая ROC AUC не гарантирует калибровку; log loss оценивает вероятностный прогноз в целом, а не только калибровку.

## 7. Протокол проверки
**Train** используем для обучения параметров модели и преобразований. **Validation** — для выбора модели, гиперпараметров и порога. **Test** — для финальной оценки после фиксации решений.

Сравниваем модели на одинаковых разбиениях. Выбираем основную метрику исходя из задачи, остальные метрики и графики используем для объяснения результата. Многократный выбор лучшего варианта по test превращает его в validation.

Перед экспериментом полезно проверить три вещи: какой результат оцениваем — метку, порядок или вероятность; что ограничивает модель — признаки или обучение; какие данные использованы для выбора параметров.